# v0.13.0 -- `SurrealFunc` & server-side values

Some values belong to the **server**, not to your process: a creation timestamp should come from the database clock, a password hash from the database's own crypto. v0.13.0 adds:

- **`SurrealFunc(expression)`** -- marks a raw SurrealQL expression to be evaluated server-side, plus **`SurrealFunc.call(fn, *args)`** to build a call from a function name.
- **`server_values=` / `extra_vars=` on `save()` and `merge()`** -- the write compiles to `CREATE $rid SET ...` / `UPDATE $rid SET ...`; functions are inlined, every other value is a **bound parameter**.
- **Six curated function-name enums** whose every member is tested against SurrealDB **2.6.5 and 3.1.3**.

**Same on SurrealDB 2.6.x and 3.x**: this feature uses no 3.x-only primitive. Only the inherited v0.9.0 transaction rule differs, and section 7 shows exactly how.

## 1. Connect

WebSocket (`.../rpc`) so native interactive transactions are used on SurrealDB 3.x. Point `SURREALDB_HOST` / `SURREALDB_PORT` at your own server to run this notebook elsewhere.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Which server line are we on?

The feature itself behaves identically on both lines. What the probe below decides is only the **transaction strategy** used in section 7: native interactive (SurrealDB 3.x) or buffered (2.6.x / HTTP).

In [2]:
import contextlib


async def native_tx_supported() -> bool:
    client = await SurrealDBConnectionManager.get_client()
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await native_tx_supported()
version = await (await SurrealDBConnectionManager.get_client()).version()
print("Server version:", version)
print("Interactive transactions available (SurrealDB 3.x):", interactive)

Server version: surrealdb-3.2.4+20260803.93ab219
Interactive transactions available (SurrealDB 3.x): True


## 3. A model + a clean slate

`joined_at`, `updated_at` and `password_hash` are never written by Python in this notebook -- the server fills them.

In [3]:
from typing import Any

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Player(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    name: str = ""
    plan: str = "free"
    score: int = 0
    joined_at: Any = None
    updated_at: Any = None
    password_hash: str = ""


client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Player;", {})
print("clean slate ready")

clean slate ready


## 4. `save(server_values=...)` -- the database clock

Passing a `SurrealFunc` makes the ORM compile `CREATE $rid SET name = $_sv_name, ..., joined_at = time::now()`. The expression reaches the server **as an expression**, so SurrealDB evaluates it. The row it returns is applied to the instance, so the computed value is readable immediately -- no extra `refresh()`.

In [4]:
from datetime import datetime

from surreal_orm_lite import SurrealFunc

player = Player(id="alice", name="Alice", score=10)
await player.save(server_values={"joined_at": SurrealFunc("time::now()")})

print("instance joined_at :", player.joined_at)
print("is a real datetime :", isinstance(player.joined_at, datetime))

rows = await client.query("SELECT * FROM Player:alice;", {})
print("stored joined_at   :", rows[0]["joined_at"])

instance joined_at : 2026-08-23 15:14:24.470732+00:00
is a real datetime : True
stored joined_at   : 2026-08-23 15:14:24.470732+00:00


Without `server_values`, the same value would have to be computed in Python -- a different clock, and a round-trip apart from the write. Note also what is *not* stored: the literal string `"time::now()"`.

## 5. `extra_vars=` -- user input stays a bound parameter

The expression is inserted into the query verbatim, so it must never be built from user input. Instead the expression **references** a parameter and you pass the value through `extra_vars`, which is bound exactly like a field value.

Here the raw password never appears in the query text, and the hash is computed by the database.

In [5]:
from surreal_orm_lite import SurrealCryptoFunction

raw_password = "correct horse battery staple"

user = Player(id="bob", name="Bob")
await user.save(
    server_values={"password_hash": SurrealFunc.call(SurrealCryptoFunction.ARGON2_GENERATE, "$password")},
    extra_vars={"password": raw_password},
)

print("hash:", user.password_hash[:60], "...")

# The database verifies it against the original password...
ok = await client.query(
    "RETURN crypto::argon2::compare($h, $p);",
    {"h": user.password_hash, "p": raw_password},
)
print("argon2 verifies   :", ok)

# ...and the raw password was never stored.
rows = await client.query("SELECT * FROM Player:bob;", {})
print("raw password kept :", raw_password in str(rows[0]))

hash: $argon2id$v=19$m=19456,t=2,p=1$HZJyidOwpQ1H1Yh/o0uxWw$IzHcNn ...
argon2 verifies   : True
raw password kept : False


A field value that *looks* like an injection stays a value, because it is bound rather than interpolated:

In [6]:
evil = "'; REMOVE TABLE Player; --"
await Player(id="mallory", name=evil).save(server_values={"joined_at": SurrealFunc("time::now()")})

rows = await client.query("SELECT * FROM Player:mallory;", {})
print("stored verbatim :", rows[0]["name"])
print("table still has :", await Player.objects().count(), "rows")

stored verbatim : '; REMOVE TABLE Player; --
table still has : 3 rows


## 6. `merge(server_values=...)` -- partial update, server-computed field

`merge()` takes the same two arguments and stays **partial**: only the listed fields change. A `server_values` entry overrides a keyword of the same name.

In [7]:
from surreal_orm_lite import SurrealTimeFunction

await player.merge(plan="pro", server_values={"updated_at": SurrealFunc.call(SurrealTimeFunction.NOW)})

print("plan       :", player.plan)
print("updated_at :", player.updated_at)

rows = await client.query("SELECT * FROM Player:alice;", {})
print("untouched score preserved :", rows[0]["score"])
print("untouched name preserved  :", rows[0]["name"])

plan       : pro
updated_at : 2026-08-23 15:14:24.560297+00:00
untouched score preserved : 10
untouched name preserved  : Alice


## 7. Inside a transaction

The two strategies commit the **same** database state. They differ only in *when* the instance can know a server-computed value:

- **interactive** (WebSocket + SurrealDB 3.x): the statement runs immediately, so the returned row updates the instance inside the block;
- **buffered** (HTTP or SurrealDB 2.6.x): the statement runs at commit, so the computed field stays at its previous value until you `refresh()`.

In [8]:
async with SurrealDBConnectionManager.transaction() as tx:
    tx_player = Player(id="carol", name="Carol")
    await tx_player.save(tx=tx, server_values={"joined_at": SurrealFunc("time::now()")})
    print("strategy                 :", "interactive" if tx.is_interactive else "buffered")
    print("instance inside the block:", tx_player.joined_at)

rows = await client.query("SELECT * FROM Player:carol;", {})
print("committed in the database:", rows[0]["joined_at"])

if not tx_player.joined_at:
    await tx_player.refresh()
    print("after refresh()           :", tx_player.joined_at)

strategy                 : interactive
instance inside the block: 2026-08-23 15:14:24.570747+00:00
committed in the database: 2026-08-23 15:14:24.570747+00:00


A rollback discards the write on both strategies:

In [9]:
with contextlib.suppress(RuntimeError):
    async with SurrealDBConnectionManager.transaction() as tx:
        await Player(id="ghost", name="Ghost").save(tx=tx, server_values={"joined_at": SurrealFunc("time::now()")})
        raise RuntimeError("abort")

print("Player:ghost exists:", await Player.objects().filter(name="Ghost").exists())

Player:ghost exists: False


## 8. The function catalog

`SurrealFunc.call` accepts a plain string or a member of the shipped enums. Every catalogued member is executed against **both** SurrealDB 2.6.5 and 3.1.3 by the test suite, so autocompletion only offers names that work on both lines.

Names that diverge between the lines are deliberately excluded (`rand::guid` is 2.6-only; `type::is::*` became `type::is_*` in 3.x) -- pass those as a plain string if you target one line.

In [10]:
from surreal_orm_lite import (
    SurrealArrayFunction,
    SurrealMathFunction,
    SurrealRandFunction,
    SurrealStringFunction,
)

for enum_cls in (SurrealTimeFunction, SurrealMathFunction, SurrealStringFunction,
                 SurrealArrayFunction, SurrealCryptoFunction, SurrealRandFunction):
    members = list(enum_cls)
    print(f"{enum_cls.__name__:<24} {len(members):>2} members  e.g. {', '.join(str(m) for m in members[:3])}")

SurrealTimeFunction      17 members  e.g. time::now, time::ceil, time::floor
SurrealMathFunction      15 members  e.g. math::abs, math::ceil, math::floor
SurrealStringFunction    14 members  e.g. string::concat, string::lowercase, string::uppercase
SurrealArrayFunction     13 members  e.g. array::append, array::concat, array::add
SurrealCryptoFunction     7 members  e.g. crypto::argon2::generate, crypto::argon2::compare, crypto::bcrypt::generate
SurrealRandFunction      11 members  e.g. rand, rand::uuid, rand::uuid::v4


In [11]:
# A couple of them in action, server-side:
uuid_player = Player(id="dave", name="Dave")
await uuid_player.save(
    server_values={
        "joined_at": SurrealFunc.call(SurrealTimeFunction.NOW),
        "password_hash": SurrealFunc.call(SurrealRandFunction.UUID_V7),
        "name": SurrealFunc.call(SurrealStringFunction.UPPERCASE, "'dave'"),
    }
)
print("name  (string::uppercase):", uuid_player.name)
print("uuid  (rand::uuid::v7)  :", uuid_player.password_hash)
print("stamp (time::now)       :", uuid_player.joined_at)

name  (string::uppercase): DAVE
uuid  (rand::uuid::v7)  : 01a02f2f-ead7-79e0-b719-c75ac39eb68b
stamp (time::now)       : 2026-08-23 15:14:24.599066+00:00


## 9. Errors you should expect

The guards are deliberate: a non-`SurrealFunc` server value would silently be stored as a literal string, and `extra_vars` with nothing to reference them is always a mistake.

In [12]:
# A server value must be a SurrealFunc, not a string that looks like one.
try:
    await Player(id="x", name="X").save(server_values={"joined_at": "time::now()"})
except TypeError as e:
    print("TypeError :", e)

# extra_vars without server_values has nothing to bind to.
try:
    await Player(id="x", name="X").save(extra_vars={"password": "p"})
except ValueError as e:
    print("ValueError:", e)

# A SurrealFunc may not chain statements.
try:
    SurrealFunc("time::now(); REMOVE TABLE Player")
except ValueError as e:
    print("ValueError:", e)

TypeError : server_values['joined_at'] must be a SurrealFunc, got 'str'. Wrap the expression: SurrealFunc('time::now()').
ValueError: extra_vars requires server_values: no SurrealFunc expression could reference them.
ValueError: SurrealFunc expression may not contain ';' (statement terminator): 'time::now(); REMOVE TABLE Player'. Pass a single expression; use extra_vars for values.


## 10. Cleanup

In [13]:
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Player;", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
